In [1]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [2]:
data = np.load("pems04.npz")

traffic = data["data"][:,:,2]

print(traffic.shape)

(16992, 307)


In [3]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

print(
    traffic_scaled.shape
)

(16992, 307)


In [4]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    -
    sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(16980, 12, 307)
(16980, 307)


In [5]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

print(X_train.shape)
print(X_test.shape)

(13584, 12, 307)
(3396, 12, 307)


In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [7]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [8]:
import torch
import torch.nn as nn

class MultiStepLSTM(nn.Module):

    def __init__(
        self,
        num_nodes=307,
        hidden_size=64
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=num_nodes,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            num_nodes
        )

    def forward(self,x):

        output, (hidden, cell) = self.lstm(x)

        hidden = hidden[-1]

        x = self.fc(hidden)

        return x

In [9]:
model = MultiStepLSTM()

X_batch, y_batch = next(iter(train_loader))

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 307])
torch.Size([64, 307])


In [10]:
model = MultiStepLSTM()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


import time

train_start = time.time()

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}",
        f"Loss: {total_loss/len(train_loader):.6f}"
    )

Epoch 1/30 Loss: 0.049122
Epoch 2/30 Loss: 0.009835
Epoch 3/30 Loss: 0.007497
Epoch 4/30 Loss: 0.006987
Epoch 5/30 Loss: 0.006561
Epoch 6/30 Loss: 0.006113
Epoch 7/30 Loss: 0.005728
Epoch 8/30 Loss: 0.005426
Epoch 9/30 Loss: 0.005215
Epoch 10/30 Loss: 0.005042
Epoch 11/30 Loss: 0.004873
Epoch 12/30 Loss: 0.004728
Epoch 13/30 Loss: 0.004567
Epoch 14/30 Loss: 0.004416
Epoch 15/30 Loss: 0.004312
Epoch 16/30 Loss: 0.004214
Epoch 17/30 Loss: 0.004131
Epoch 18/30 Loss: 0.004027
Epoch 19/30 Loss: 0.003936
Epoch 20/30 Loss: 0.003851
Epoch 21/30 Loss: 0.003773
Epoch 22/30 Loss: 0.003689
Epoch 23/30 Loss: 0.003621
Epoch 24/30 Loss: 0.003538
Epoch 25/30 Loss: 0.003469
Epoch 26/30 Loss: 0.003408
Epoch 27/30 Loss: 0.003358
Epoch 28/30 Loss: 0.003305
Epoch 29/30 Loss: 0.003259
Epoch 30/30 Loss: 0.003224


In [11]:
train_time = time.time() - train_start

print("Time Taken:", train_time)

Time Taken: 44.18774342536926


In [12]:
torch.save(
    model.state_dict(),
    "MultiStepLSTM-PEMS04.pth"
)

In [13]:
model.eval()

with torch.no_grad():

    predictions = model(
        X_test
    )

predictions = predictions.numpy()

true_values = y_test.numpy()

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.039732546
RMSE: 0.0670343


In [14]:
from sklearn.metrics import r2_score

mape = np.mean(
    np.abs((true_values - predictions) /
           np.maximum(np.abs(true_values), 1e-6))
) * 100

r2 = r2_score(
    true_values.flatten(),
    predictions.flatten()
)
print("MAPE:", mape)
print("R2:", r2)

MAPE: 5755.37109375
R2: 0.7863175485025956


In [15]:
params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", params)

Parameters: 115443
